# Bài 46: Kiến trúc Transformer (The Transformer Architecture)

Bài 45 đã xây dựng một phép toán chú ý. Một **Transformer** (<a href="../references.html#vaswani-2017">Vaswani et al., 2017</a><span class="landmark-paper">&#9733;</span>) bọc phép toán đó thành một khối dùng lại được và xếp chồng nhiều khối như vậy. Bài này bổ sung ba thành phần mà riêng cơ chế chú ý còn thiếu: **nhiều đầu chú ý (multiple attention heads)** (để một lớp có thể theo dõi nhiều loại quan hệ cùng lúc), **mã hóa vị trí (positional encoding)** (để thứ tự không vô hình đối với một cơ chế vốn coi chuỗi như một tập không có thứ tự), và **khối mã hóa (encoder block)** (chú ý cộng với một mạng truyền thẳng (feedforward) theo từng vị trí, được nối với nhau bằng các kết nối dư và chuẩn hóa).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Chú ý đa đầu

Một phép toán chú ý đơn lẻ tính một mẫu tương đồng giữa mọi cặp vị trí. **Chú ý đa đầu (multi-head attention)** chia số chiều mô hình thành vài phần nhỏ hơn (các đầu), chạy chú ý độc lập trong mỗi phần, và nối các kết quả lại — cho phép các đầu khác nhau chuyên biệt hóa vào các loại quan hệ khác nhau (ví dụ một đầu theo dõi các vị trí lân cận, một đầu khác theo dõi một phụ thuộc tầm xa cụ thể) thay vì lấy trung bình mọi thứ thành một mẫu duy nhất.

In [ ]:
D, H = 16, 4  # model dimension, number of heads
mha_torch = nn.MultiheadAttention(D, H, batch_first=True)

def multihead_attention(x, mha):
    B, T, _ = x.shape
    d_head = D // H
    Wq, Wk, Wv = mha.in_proj_weight.chunk(3, dim=0)
    bq, bk, bv = mha.in_proj_bias.chunk(3, dim=0)
    Q = (x @ Wq.T + bq).view(B, T, H, d_head).transpose(1, 2)
    K = (x @ Wk.T + bk).view(B, T, H, d_head).transpose(1, 2)
    V = (x @ Wv.T + bv).view(B, T, H, d_head).transpose(1, 2)
    scores = Q @ K.transpose(-2, -1) / np.sqrt(d_head)
    weights = F.softmax(scores, dim=-1)
    out = (weights @ V).transpose(1, 2).reshape(B, T, D)
    return out @ mha.out_proj.weight.T + mha.out_proj.bias

torch.manual_seed(0)
x = torch.randn(2, 5, D)
out_manual = multihead_attention(x, mha_torch)
out_torch, _ = mha_torch(x, x, x, need_weights=False)

print(f'model dim = {D}, heads = {H}, dim per head = {D // H}')
print(f'max abs diff vs nn.MultiheadAttention: {(out_manual - out_torch).abs().max().item():.2e}')

## Mã hóa vị trí

Chú ý tính độ tương đồng giữa các vector nội dung — không có gì trong công thức ở Bài 45 nhắc tới việc một vị trí nằm *ở đâu* trong chuỗi. Hoán vị chuỗi đầu vào, và đầu ra của chú ý hoán vị theo, giống hệt: về bản chất nó là một phép toán trên tập hợp, mù với thứ tự. **Mã hóa vị trí** khắc phục điều này bằng cách cộng một mẫu duy nhất, xác định vào mỗi vị trí trước khi chú ý chạy, để vị trí trở thành một phần của thứ được so sánh. Lựa chọn của bài báo Transformer gốc là một hình sin cố định (không học được) ở nhiều tần số:

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/D}), \qquad PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/D})$$

In [ ]:
def positional_encoding(T, D):
    pos = torch.arange(T).unsqueeze(1).float()
    i = torch.arange(D).unsqueeze(0).float()
    angle_rates = 1.0 / (10000 ** (2 * (i // 2) / D))
    angles = pos * angle_rates
    pe = torch.zeros(T, D)
    pe[:, 0::2] = torch.sin(angles[:, 0::2])
    pe[:, 1::2] = torch.cos(angles[:, 1::2])
    return pe

pe = positional_encoding(50, 32)
plt.figure(figsize=(6, 4))
plt.imshow(pe.numpy().T, cmap='RdBu', aspect='auto')
plt.xlabel('position'); plt.ylabel('encoding dimension')
plt.title('Sinusoidal positional encoding')
plt.colorbar(fraction=0.046)
plt.show()

### Điều này có thực sự quan trọng không? Một nhiệm vụ đòi hỏi biết thứ tự

Hãy dựng một nhiệm vụ không thể giải chỉ từ nội dung: một vector cố định `A` và một vector cố định `B` xuất hiện ở hai vị trí ngẫu nhiên trong một chuỗi nhiễu, và nhãn đơn giản là "`A` có xuất hiện trước `B` không?" Một mô hình gộp trung bình đầu ra chú ý trên chuỗi chỉ có thể biết *hai token nào có mặt*, không bao giờ biết thứ tự của chúng, trừ khi vị trí được đưa vào bằng cách nào đó.

In [ ]:
T = 8
vec_a, vec_b = torch.randn(D), torch.randn(D)

def make_order_dataset(rng, n):
    X, y = [], []
    for _ in range(n):
        pos_a, pos_b = rng.choice(T, size=2, replace=False)
        seq = torch.randn(T, D) * 0.1
        seq[pos_a] = vec_a
        seq[pos_b] = vec_b
        X.append(seq)
        y.append(1.0 if pos_a < pos_b else 0.0)
    return torch.stack(X), torch.tensor(y, dtype=torch.float32)

rng = np.random.default_rng(3)
X_train, y_train = make_order_dataset(rng, 400)
X_test, y_test = make_order_dataset(rng, 150)

class TinyAttnClassifier(nn.Module):
    def __init__(self, use_pos_enc):
        super().__init__()
        self.use_pos_enc = use_pos_enc
        self.mha = nn.MultiheadAttention(D, 4, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(D, 16), nn.ReLU(), nn.Linear(16, 1))
        if use_pos_enc:
            self.register_buffer('pe', positional_encoding(T, D))

    def forward(self, x):
        if self.use_pos_enc:
            x = x + self.pe
        attn_out, _ = self.mha(x, x, x, need_weights=False)
        return self.fc(attn_out.mean(dim=1)).squeeze(-1)

def train_eval(use_pos_enc, seed, epochs=300, lr=0.01):
    torch.manual_seed(seed)
    model = TinyAttnClassifier(use_pos_enc)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(X_train), y_train)
        loss.backward()
        opt.step()
    with torch.no_grad():
        return ((model(X_test) > 0).float() == y_test).float().mean().item()

no_pe_accs = [train_eval(False, seed) for seed in range(5)]
pe_accs = [train_eval(True, seed) for seed in range(5)]

print(f'without positional encoding: mean test acc = {np.mean(no_pe_accs):.1%}  (+/- {np.std(no_pe_accs):.1%})')
print(f'with positional encoding:    mean test acc = {np.mean(pe_accs):.1%}  (+/- {np.std(pe_accs):.1%})')

Không có mã hóa vị trí, mô hình kẹt ở mức đoán ngẫu nhiên — nó chỉ có thể báo cáo hai token nào đã xuất hiện, không bao giờ biết thứ tự tương đối của chúng, bất kể huấn luyện bao lâu. Thêm hình sin cố định cho mỗi vị trí một dấu hiệu riêng biệt mà mô hình có thể dựa vào, và nhiệm vụ trở nên tầm thường.

## Khối mã hóa

Một lớp Transformer đơn lẻ gồm chú ý đa đầu, một mạng truyền thẳng nhỏ theo từng vị trí, và hai kết nối dư (Bài 37) với chuẩn hóa theo lớp (layer normalization) (chuẩn hóa theo lô ở Bài 37, nhưng chuẩn hóa theo chiều *đặc trưng* cho từng token riêng lẻ thay vì theo lô):

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, D, H, FF):
        super().__init__()
        self.mha = nn.MultiheadAttention(D, H, batch_first=True)
        self.ln1 = nn.LayerNorm(D)
        self.ff = nn.Sequential(nn.Linear(D, FF), nn.ReLU(), nn.Linear(FF, D))
        self.ln2 = nn.LayerNorm(D)

    def forward(self, x):
        attn_out, _ = self.mha(x, x, x, need_weights=False)
        x = self.ln1(x + attn_out)   # residual + norm around attention
        ff_out = self.ff(x)
        x = self.ln2(x + ff_out)     # residual + norm around the feedforward network
        return x

FF = 32
block = EncoderBlock(D, H, FF)
layer_torch = nn.TransformerEncoderLayer(d_model=D, nhead=H, dim_feedforward=FF,
                                          batch_first=True, dropout=0.0)
layer_torch.eval()

# copy torch's weights into our block so the two are directly comparable
block.mha.load_state_dict(layer_torch.self_attn.state_dict())
block.ln1.load_state_dict(layer_torch.norm1.state_dict())
block.ln2.load_state_dict(layer_torch.norm2.state_dict())
block.ff[0].load_state_dict({'weight': layer_torch.linear1.weight, 'bias': layer_torch.linear1.bias})
block.ff[2].load_state_dict({'weight': layer_torch.linear2.weight, 'bias': layer_torch.linear2.bias})

x2 = torch.randn(2, 6, D)
out_block = block(x2)
out_layer = layer_torch(x2)
print(f'encoder block max abs diff vs nn.TransformerEncoderLayer: {(out_block - out_layer).abs().max().item():.2e}')
print(f'output shape unchanged from input: {tuple(out_block.shape)} == {tuple(x2.shape)}')

Đầu ra của khối có chính xác cùng hình dạng với đầu vào — cùng mẹo như khối dư ở Bài 37, và vì cùng lý do: điều đó có nghĩa là các khối có thể được xếp chồng sâu tùy ý, mỗi khối tinh chỉnh cùng một chuỗi vector, mà không cần đổi hình dạng giữa chúng. Một *bộ mã hóa (encoder)* Transformer đầy đủ chỉ là `N` bản sao của khối này xếp chồng nối tiếp (`nn.TransformerEncoder` trong PyTorch); một *bộ giải mã (decoder)* thêm một bước chú ý thứ hai trong mỗi khối chú ý tới đầu ra của bộ mã hóa, cộng với một mặt nạ nhân quả (Bài 45) trên phép tự chú ý (self-attention) của chính nó. Vision Transformer (Bài 47) dùng lại phía bộ mã hóa gần như không thay đổi — khác biệt thực sự duy nhất là thứ được đưa vào làm chuỗi vector ban đầu.

### Bài tập

1. Đổi `H` (số đầu) từ 4 thành 1, giữ cố định `D=16`, và chạy lại phép đối chiếu chú ý đa đầu. Với một đầu duy nhất, có còn khác biệt đáng kể nào so với hàm `attention` một đầu ở Bài 45 không?
2. Trong nhiệm vụ phát hiện thứ tự, đổi `T` (độ dài chuỗi) từ 8 thành 32. Độ chính xác của mô hình có mã hóa vị trí có giữ vững không, hay chuỗi dài hơn làm nhiệm vụ khó hơn theo cách mà riêng mã hóa vị trí không khắc phục được?
3. `EncoderBlock` ở trên dùng "post-norm" (`LayerNorm` áp dụng *sau* phép cộng dư, khớp với bài báo gốc năm 2017). Nhiều Transformer hiện đại thay vào đó dùng "pre-norm": `x = x + self.mha(self.ln1(x))`. Hãy cài đặt pre-norm và so sánh hai cách ở độ sâu lớn hơn (xếp chồng 10 khối) — có cách nào huấn luyện ổn định hơn không, lặp lại câu chuyện gradient biến mất ở Bài 37?